### Reading csv file from catalog

In [0]:
df=spark.read.csv("/Volumes/day1/default/sample1",header=True,inferSchema=True)

In [0]:
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,blank,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,Blank
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,blank,450,C


In [0]:
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- JoinDate: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Category: string (nullable = true)



## 1. Handle "blank"/"Blank" values - NULL


In [0]:
df=df.replace(["blank","Blank"],None)

In [0]:
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


### 2. Fix country column (Standarization) india/USA - inconsistent


In [0]:
from pyspark.sql.functions import upper
df=df.withColumn("Country",upper(col("Country")))
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,INDIA,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,NEW YORK,03-15-2023,400,B
107,Trent,INDIA,10-04-2023,350,B
108,Bob,INDIA,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


In [0]:
df=df.withColumn("joinDate",when(col("joinDate").isNull(),None).otherwise(col("joinDate")))

In [0]:
df.display()

CustomerID,Name,Country,joinDate,Sales,Category
101,Alice,USA,2023-01-15,250,A
102,Bob,INDIA,2023-01-01,150,B
103,Charlie,UK,2023-01-20,null,C
104,Alice,USA,2023-01-15,250,A
105,Eve,UK,2023-01-01,300,null
106,Mallory,NEW YORK,2023-01-03,400,B
107,Trent,INDIA,2023-01-10,350,B
108,Bob,INDIA,2023-01-05,150,B
109,Oscar,USA,2023-01-28,500,A
110,Peggy,UK,null,450,C


In [0]:
df=df.fillna({
    "Category":"Unknown",
    "Sales":"0"
})
df.display()

CustomerID,Name,Country,joinDate,Sales,Category
101,Alice,USA,2023-01-15,250,A
102,Bob,INDIA,2023-01-01,150,B
103,Charlie,UK,2023-01-20,0,C
104,Alice,USA,2023-01-15,250,A
105,Eve,UK,2023-01-01,300,Unknown
106,Mallory,NEW YORK,2023-01-03,400,B
107,Trent,INDIA,2023-01-10,350,B
108,Bob,INDIA,2023-01-05,150,B
109,Oscar,USA,2023-01-28,500,A
110,Peggy,UK,null,450,C


In [0]:
df=df.filter(col("Sales")==0)
df.display()

CustomerID,Name,Country,joinDate,Sales,Category
103,Charlie,UK,2023-01-20,0,C
